# Pipeline de données XAUUSD · phase 1

**Périmètre** · Données existantes 2004–2026, exports complémentaires 2023–2026, conservation Bronze et premier audit.

Exécuter les cellules dans l’ordre. Le mode par défaut affiche les preuves et résultats enregistrés le 24 novembre 2026. La dernière section présente les essais déjà documentés et le prochain chantier de nettoyage et de fusion.

[Sources originales](https://drive.google.com/drive/folders/1a_HmvMG4HgrjlRCYAx0PYwyJrZNangCr) · [Projet GitHub](https://github.com/hamzaelattar/Multi-Agents-Trading)

## Configuration

- **snapshot** · Affiche l’audit sauvegardé et vérifie la présence des fichiers quand le dossier `data` est disponible.
- **comparaison** · Recalcule le chevauchement des CSV historiques avec DuckDB.
- **complet** · Recalcule le profil des deux CSV historiques et leur chevauchement. Prévoir plusieurs gigaoctets temporaires.

Les tableaux d’archives complémentaires présentent les audits déjà réalisés. Le fichier doit garder son nom ou être indiqué par `XAUUSD_NOTEBOOK_PATH` si le noyau ne le trouve pas.

In [1]:
import base64
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from contextlib import chdir
import csv
from datetime import date, timedelta
import io
from itertools import count
import json
import os
from pathlib import Path
import subprocess
from tempfile import TemporaryDirectory
import zlib

from IPython.display import Markdown, display

mode = os.environ.get('XAUUSD_MODE', 'snapshot')  # snapshot / comparaison / complet
if mode not in {'snapshot', 'comparaison', 'complet'}:
    raise ValueError(f'Mode inconnu : {mode}')
racines = (Path.cwd(), *Path.cwd().parents, Path.home() / 'OneDrive/Desktop/Multi-Agents-Trading')
racine_detectee = next((p for p in racines if (p / 'pipelinebigdata').is_dir()), Path.cwd())
racine = Path(os.environ.get('XAUUSD_PROJECT_DIR', racine_detectee)).resolve()
data_dir = Path(os.environ.get('XAUUSD_DATA_DIR', racine / 'data')).resolve()
duckdb = Path(os.environ.get('DUCKDB_EXE', Path(os.environ.get('LOCALAPPDATA', '.')) / 'duckdb/cli/2.0.1/duckdb.exe')).resolve()
scratch = Path(os.environ.get('XAUUSD_SCRATCH_DIR', racine / '.profilage_temp')).resolve()
FILES = {
    'ancien': 'XAUUSD_Ticks_2004.01.01_2021.10.15.csv',
    'recent': 'XAUUSD_Ticks_2021.01.04_2026.11.20.csv',
}
HEADER = b'Time (EET),Ask,Bid,AskVolume,BidVolume'
NUMBER = r'[+-]?[0-9]+,[0-9]+'
RAW_FORMAT = r'[0-9]{4}\.[0-9]{2}\.[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}\.[0-9]{3},' + ','.join([NUMBER] * 4)
VALUES = {'ask': 2, 'bid': 4, 'ask_volume': 6, 'bid_volume': 8}



def fr(value):
    if value is None:
        return '—'
    if isinstance(value, int):
        return f'{value:,}'.replace(',', ' ')
    if isinstance(value, float):
        return f'{value:.8g}'.replace('.', ',')
    return str(value).replace('|', r'\|').replace('\n', ' ')


def tableau(headers, rows):
    values = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    values.extend('| ' + ' | '.join(fr(x) for x in row) + ' |' for row in rows)
    display(Markdown('\n'.join(values)))


tableau(['Configuration', 'Valeur'], [
    ('Mode', mode), ('Dossier CSV', data_dir.name), ('DuckDB disponible', 'oui' if duckdb.is_file() else 'non'),
    *((kind.capitalize(), f'{name} · ' + ('présent' if (data_dir / name).is_file() else 'absent')) for kind, name in FILES.items()),
])

def afficher(texte):
    display(Markdown(texte))


nom_notebook = 'Profilage_donnees_XAUUSD.ipynb'
chemin_impose = os.environ.get('XAUUSD_NOTEBOOK_PATH')
candidats = [Path(chemin_impose)] if chemin_impose else [
    racine / 'pipelinebigdata/notebooks' / nom_notebook, Path.cwd() / nom_notebook,
]
for candidat in candidats:
    if candidat.is_file():
        meta_audit = json.loads(candidat.read_text(encoding='utf-8-sig'))['metadata'].get('xauusd', {})
        if 'pipeline' in meta_audit:
            break
else:
    raise FileNotFoundError(f'Notebook source introuvable. Indiquer son chemin avec XAUUSD_NOTEBOOK_PATH.')

preuves = meta_audit['pipeline']
afficher(f"Ce mode détermine quelles mesures sont recalculées. Les preuves enregistrées datent du **{preuves['date_audit']}**.")


| Configuration | Valeur |
| --- | --- |
| Mode | snapshot |
| Dossier CSV | data |
| DuckDB disponible | oui |
| Ancien | XAUUSD_Ticks_2004.01.01_2021.10.15.csv · présent |
| Recent | XAUUSD_Ticks_2021.01.04_2026.11.20.csv · présent |

Ce mode détermine quelles mesures sont recalculées. Les preuves enregistrées datent du **24 novembre 2026**.

## Données existantes · 2004–2026

In [2]:
tableau(['CSV historique', 'Présence locale'], [
    (fichier, 'Présent' if (data_dir / fichier).is_file() else 'Hors du dossier local')
    for fichier in FILES.values()
])
afficher("Les noms des CSV indiquent les bornes observées à l’audit. Les dates des CSV historiques suivent l’horloge exportée EET/EEST.")

| CSV historique | Présence locale |
| --- | --- |
| XAUUSD_Ticks_2004.01.01_2021.10.15.csv | Présent |
| XAUUSD_Ticks_2021.01.04_2026.11.20.csv | Présent |

Les noms des CSV indiquent les bornes observées à l’audit. Les dates des CSV historiques suivent l’horloge exportée EET/EEST.

## Nouveaux exports · 2023–2026

In [3]:
tableau(['Année', 'Source', 'Ticks audités', 'Premier UTC', 'Dernier UTC', 'Dossier local'], [
    (lot['year'], lot['source'], lot['ticks'], lot['first'], lot['last'],
     'Présent' if (data_dir / lot['directory']).is_dir() else 'Hors du dossier local')
    for lot in preuves['recent']
])
lot_jforex = preuves['jforex']
afficher(f"Ces archives contiennent les prix Bid et Ask et les deux volumes. Le lot JForex du **23 novembre 2026** apporte **{fr(lot_jforex['ticks'])} ticks** sur **{lot_jforex['hours']} heures**.")

| Année | Source | Ticks audités | Premier UTC | Dernier UTC | Dossier local |
| --- | --- | --- | --- | --- | --- |
| 2 023 | Carlos Silva | 36 414 968 | 2023-01-02 23:00:04.27 | 2023-12-29 21:58:59.074 | Présent |
| 2 024 | Touhid-HZ | 56 217 145 | 2024-01-01 23:00:00.312 | 2024-12-31 21:58:55.257 | Présent |
| 2 025 | Touhid-HZ | 71 403 972 | 2025-01-01 23:00:00.455 | 2025-12-31 21:58:59.943 | Présent |
| 2 026 | Touhid-HZ | 84 371 369 | 2026-01-01 23:00:00.596 | 2026-11-22 23:59:59.140 | Présent |

Ces archives contiennent les prix Bid et Ask et les deux volumes. Le lot JForex du **23 novembre 2026** apporte **500 254 ticks** sur **15 heures**.

## BRONZE · Raw Data

In [4]:
tableau(['Archive d’origine', 'CSV correspondant', 'SHA-256 consigné', 'Présence locale'], [
    (lot['archive'], lot['matching_csv'], lot['member_sha256'],
     'Présente' if (data_dir / lot['archive']).is_file() else 'Hors du dossier local')
    for lot in preuves['archives']
])
afficher(f"Bronze conserve les fichiers reçus et leur provenance. Les **{fr(preuves['histdata_months'])} mois HistData** totalisent **{fr(preuves['histdata_ticks'])} lignes brutes**. HistData fournit des prix Bid et Ask, avec volumes de cotation indisponibles.")

| Archive d’origine | CSV correspondant | SHA-256 consigné | Présence locale |
| --- | --- | --- | --- |
| archives/XAUUSD_Ticks_2004.01.01_2021.10.15.zip | XAUUSD_Ticks_2004.01.01_2021.10.15.csv | 956b6b261e9dba1f2b89cc03667ef6cd0db509cdbc920f97f2a4d4c95159da2f | Présente |
| archives/XAUUSD_Ticks_2021.01.04_2026.11.20.rar | XAUUSD_Ticks_2021.01.04_2026.11.20.csv | 0faf9bb44ca70ae9b088ce864fe27fdded3515a0604f26403c8424f8b430d5d5 | Présente |

Bronze conserve les fichiers reçus et leur provenance. Les **194 mois HistData** totalisent **472 336 326 lignes brutes**. HistData fournit des prix Bid et Ask, avec volumes de cotation indisponibles.

## Data Profiling · premier audit

Les contrôles ci-dessous portent sur les deux CSV historiques.

## Moteur DuckDB intégré

La lecture conserve d’abord chaque ligne en texte brut. Les gros fichiers sont découpés aux fins de ligne. Le dernier tick du bloc précédent est transmis au suivant pour vérifier l’ordre et calculer les intervalles aux jonctions.

In [5]:
def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def number_fields():
    for name, position in VALUES.items():
        whole = f'parts[{position}]'
        fraction = f'parts[{position + 1}]'
        yield (f"CASE WHEN standard OR (width = 9 AND regexp_full_match({whole}, '[+-]?[0-9]+') "
               f"AND regexp_full_match({fraction}, '[0-9]+')) THEN "
               f"try_cast({whole} || '.' || {fraction} AS DOUBLE) END AS {name}")
        yield f"width = 9 AND {whole} = '' AND {fraction} = '' AS empty_{name}"
        yield f"width = 9 AND ({whole} = '' OR {fraction} = '') AS empty_component_{name}"


def sql_rows(exe, query, threads=3, memory_mb=3500):
    command = f"SET threads={threads}; SET memory_limit='{memory_mb}MB'; SET preserve_insertion_order=true;\n{query}"
    result = subprocess.run([str(exe), '-batch', '-bail', '-csv'], input=command,
                            encoding='utf-8', capture_output=True, check=False)
    if result.returncode:
        raise RuntimeError(result.stderr.strip())
    return list(csv.DictReader(io.StringIO(result.stdout)))


def physical_layout(path):
    with path.open('rb') as stream:
        if stream.readline().removeprefix(b'\xef\xbb\xbf').rstrip(b'\r\n') != HEADER:
            raise ValueError(f'Unexpected header: {path}')
        complete = 0
        last = b''
        while block := stream.read(16 * 1024**2):
            complete += block.count(b'\n')
            last = block
    fragment = last.rsplit(b'\n', 1)[-1]
    return complete, fragment.decode('ascii')


def stage(exe, source, target, fragment, *, preceding_rows=0, has_predecessor=False, threads=4):
    fields = ',\n'.join(number_fields())
    last = f'raw = {sql_string(fragment)}' if fragment else 'false'
    source_offset = preceding_rows - has_predecessor
    sql_rows(exe, f"""
SET partitioned_write_flush_threshold=65536;
SET partitioned_write_max_open_files=16;
COPY (
    WITH raw_rows AS (
        SELECT raw, row_number() OVER () AS row_id,
               string_split(raw, ',') AS parts
        FROM read_csv({sql_string(source.as_posix())}, columns={{'raw':'VARCHAR'}},
                      delim='|', quote='', escape='', header=true,
                      auto_detect=false, parallel={str(threads > 1).lower()})
    ), parsed AS (
        SELECT *, len(parts) AS width, {last} AS truncated,
               try_strptime(parts[1], '%Y.%m.%d %H:%M:%S.%g') AS t,
               regexp_full_match(raw, {sql_string(RAW_FORMAT)}) AS standard
        FROM raw_rows
    ), typed AS (
        SELECT raw, row_id + {source_offset} AS row_id, width, truncated, t,
               lag(t) OVER () AS previous_t,
               parts[1] = '' OR parts[1] IS NULL AS empty_time,
               standard, CASE WHEN t IS NULL THEN 'invalide' ELSE replace(left(parts[1], 7), '.', '-') END AS mois,
               {fields}
        FROM parsed
    )
    SELECT * FROM typed WHERE row_id > {preceding_rows}
) TO {sql_string(target.as_posix())} (FORMAT parquet, PARTITION_BY (mois), COMPRESSION zstd);
""", threads=threads, memory_mb=1100 if threads == 1 else 3500)


def stage_large(exe, source, target, fragment, block_size=384 * 1024**2):
    """Split on line endings, keeping a predecessor for each source-order boundary."""
    target.mkdir()

    def process_shard(path, output, preceding_rows, has_predecessor):
        stage(exe, path, output, fragment, preceding_rows=preceding_rows,
              has_predecessor=has_predecessor, threads=1)
        path.unlink()
        return output

    def collect(pending):
        done, pending = wait(pending, return_when=FIRST_COMPLETED)
        for future in done:
            output = future.result()
            for month in output.iterdir():
                destination = target / month.name
                destination.mkdir(exist_ok=True)
                for parquet in month.glob('*.parquet'):
                    parquet.rename(destination / f'{output.name}_{parquet.name}')
                month.rmdir()
            output.rmdir()
        return pending

    with source.open('rb') as stream, ThreadPoolExecutor(max_workers=4) as executor:
        header = stream.readline()
        offset = 0
        previous = b''
        remainder = b''
        pending = set()
        for index in count(1):
            read = stream.read(block_size)
            if not read and not remainder:
                break
            block = remainder + read
            end = block.rfind(b'\n') + 1
            if read:
                remainder = block[end:]
                block = block[:end]
            else:
                remainder = b''
            if not block:
                continue
            path = target.parent / f'shard_{index:03d}.csv'
            output = target.parent / f'shard_{index:03d}'
            with path.open('wb') as file:
                file.write(header)
                file.write(previous)
                file.write(block)
            pending.add(executor.submit(process_shard, path, output, offset, bool(previous)))
            if read:
                offset += block.count(b'\n')
                before_last = block.rfind(b'\n', 0, len(block) - 1)
                previous = block[before_last + 1:]
            if len(pending) >= 6:
                pending = collect(pending)
            if index % 8 == 0:
                print(f'{source.name}: {offset:,} rows dispatched to shards', flush=True)
        while pending:
            pending = collect(pending)


def as_number(value):
    if value == '':
        return None
    try:
        return int(value)
    except ValueError:
        try:
            return float(value)
        except ValueError:
            return value


def typed_rows(exe, statement):
    return [{key: as_number(val) for key, val in row.items()} for row in sql_rows(exe, statement)]


print('Moteur intégré chargé : lecture brute, validation du format et partitions DuckDB temporaires.')

Moteur intégré chargé : lecture brute, validation du format et partitions DuckDB temporaires.


## Calcul et résultats intégrés

Le mode rapide lit l’audit intégré dans les métadonnées du notebook. Si le fichier est renommé ou ouvert depuis un autre dossier de travail, `XAUUSD_NOTEBOOK_PATH` permet d’indiquer son chemin. Le mode complet recalcule les profils à partir des CSV et les conserve dans `profils` pour les cellules suivantes.

In [6]:
def counts_summary(exe, table):
    details = []
    for name in VALUES:
        details.extend([
            f'count({name}) AS {name}_lisible',
            f'count(*) FILTER (WHERE empty_{name}) AS {name}_vide',
            f'count(*) FILTER (WHERE empty_component_{name}) AS {name}_composant_vide',
            f'count(*) FILTER (WHERE width = 9 AND {name} IS NULL) AS {name}_non_lisible_structure_standard',
        ])
    return typed_rows(exe, f"""
SELECT count(*) AS lignes_lues,
       count(*) FILTER (WHERE truncated) AS finales_tronquees,
       count(*) FILTER (WHERE NOT truncated AND t IS NULL) AS horodatages_invalides,
       count(*) FILTER (WHERE NOT truncated AND empty_time) AS horodatages_vides,
       count(*) FILTER (WHERE NOT truncated AND standard) AS format_standard,
       count(*) FILTER (WHERE NOT truncated AND NOT standard) AS format_non_standard,
       min(t) FILTER (WHERE NOT truncated) AS premier_tick,
       max(t) FILTER (WHERE NOT truncated) AS dernier_tick,
       count(*) FILTER (WHERE NOT truncated AND t < previous_t) AS reculs_adjacent,
       count(*) FILTER (WHERE NOT truncated AND t = previous_t) AS egalites_adjacentes,
       count(*) FILTER (WHERE NOT truncated AND t IS NOT NULL AND previous_t IS NOT NULL) AS paires_adjacentes_lisibles,
       count(*) FILTER (WHERE NOT truncated AND t < previous_t AND cast(t AS DATE) <> cast(previous_t AS DATE)) AS reculs_changement_date,
       min(date_diff('millisecond', previous_t, t)) FILTER (WHERE NOT truncated) AS pire_recul_ms
FROM {table};
""")[0] | typed_rows(exe, f"""
SELECT {', '.join(details)},
       count(*) FILTER (WHERE ask IS NOT NULL AND bid IS NOT NULL) AS paires_de_prix_lisibles,
       count(*) FILTER (WHERE ask <= 0) AS ask_non_positifs,
       count(*) FILTER (WHERE bid <= 0) AS bid_non_positifs,
       count(*) FILTER (WHERE ask <= 0 OR bid <= 0) AS ticks_prix_non_positifs,
       count(*) FILTER (WHERE ask < bid) AS spread_negatif,
       count(*) FILTER (WHERE ask = bid) AS spread_nul,
       min(ask - bid) AS spread_minimum,
       min(ask) AS ask_minimum, max(ask) AS ask_maximum,
       min(bid) AS bid_minimum, max(bid) AS bid_maximum
FROM {table} WHERE NOT truncated;
""")[0]


def volume_stats(exe, table, column):
    return typed_rows(exe, f"""
WITH frequencies AS (
    SELECT {column} AS v, count(*) AS n FROM {table}
    WHERE NOT truncated AND {column} IS NOT NULL GROUP BY 1
), ranked AS (
    SELECT *, sum(n) OVER () AS population,
           sum(n) OVER (ORDER BY v ROWS UNBOUNDED PRECEDING) AS cumul
    FROM frequencies
)
SELECT coalesce(sum(n), 0) AS observations, coalesce(sum(n) FILTER (WHERE v < 0),0) AS negatifs,
       coalesce(sum(n) FILTER (WHERE v = 0),0) AS zeros, min(v) AS minimum,
       sum(v*n)/sum(n) AS moyenne,
       sqrt(greatest(0, sum(v*v*n)/sum(n) - pow(sum(v*n)/sum(n),2))) AS ecart_type,
       max(v) AS maximum,
       min(v) FILTER (WHERE cumul*100 >= population) AS p01,
       min(v) FILTER (WHERE cumul*4 >= population) AS p25,
       min(v) FILTER (WHERE cumul*2 >= population) AS p50,
       min(v) FILTER (WHERE cumul*4 >= population*3) AS p75,
       min(v) FILTER (WHERE cumul*20 >= population*19) AS p95,
       min(v) FILTER (WHERE cumul*100 >= population*99) AS p99,
       coalesce(sum(n) FILTER (WHERE v > 0 AND v <= 0.0001),0) AS positif_jusque_00001,
       coalesce(sum(n) FILTER (WHERE v > 0.0001 AND v <= 0.001),0) AS de_00001_a_0001,
       coalesce(sum(n) FILTER (WHERE v > 0.001 AND v <= 0.01),0) AS de_0001_a_001,
       coalesce(sum(n) FILTER (WHERE v > 0.01 AND v <= 0.1),0) AS de_001_a_01,
       coalesce(sum(n) FILTER (WHERE v > 0.1),0) AS superieur_a_01
FROM ranked;
""")[0]


def per_calendar(daily, period):
    by_day = {date.fromisoformat(row['jour']): row['ticks'] for row in daily}
    first, last = min(by_day), max(by_day)
    periods = {}
    for offset in range((last - first).days + 1):
        day = first + timedelta(days=offset)
        key = day.isoformat()[:period]
        bucket = periods.setdefault(key, {'periode': key, 'ticks': 0, 'jours_couverts': 0,
                                         'jours_dans_fenetre': 0, 'lun_ven_sans_ticks': 0})
        count = by_day.get(day, 0)
        bucket['jours_dans_fenetre'] += 1
        bucket['ticks'] += count
        bucket['jours_couverts'] += count > 0
        bucket['lun_ven_sans_ticks'] += count == 0 and day.weekday() < 5
    return list(periods.values())


def duplicates(exe, partitions):
    rows = []
    for folder in sorted(partitions.glob('mois=*')):
        table = f"read_parquet({sql_string((folder / '*.parquet').as_posix())}, hive_partitioning=false)"
        row = typed_rows(exe, f"""
WITH distinct_rows AS (
    SELECT t, raw, count(*) AS n FROM {table} WHERE NOT truncated GROUP BY t, raw
), per_timestamp AS (
    SELECT t, sum(n) AS ticks, count(*) AS variants,
           sum(n - 1) AS repetitions, count(*) FILTER (WHERE n > 1) AS groupes
    FROM distinct_rows GROUP BY t
)
SELECT coalesce(sum(repetitions),0) AS copies_identiques_excedentaires,
       coalesce(sum(groupes),0) AS groupes_lignes_identiques,
       count(*) FILTER (WHERE t IS NOT NULL AND ticks > 1) AS horodatages_repetes,
       count(*) FILTER (WHERE t IS NOT NULL AND variants > 1) AS horodatages_avec_lignes_distinctes
FROM per_timestamp;
""")[0]
        rows.append({'mois': folder.name[5:]} | row)
    keys = rows[0].keys() - {'mois'}
    return {'totaux': {key: sum(row[key] for row in rows) for key in sorted(keys)}, 'par_mois': rows}


def summarize(exe, partitions, source, physical_complete, fragment):
    table = f"read_parquet({sql_string((partitions / '*/*.parquet').as_posix())}, hive_partitioning=false)"
    checks = counts_summary(exe, table)
    checks['lignes_physiques_donnees'] = physical_complete + bool(fragment)
    checks['lignes_terminees'] = physical_complete
    checks['lignes_ignorees_par_lecteur'] = physical_complete + bool(fragment) - checks['lignes_lues']
    widths = typed_rows(exe, f"""
SELECT width AS composants, count(*) AS lignes, min(raw) AS exemple,
       count(*) FILTER (WHERE starts_with(raw, ',') OR contains(raw, ',,') OR ends_with(raw, ',')) AS lignes_avec_composant_brut_vide
FROM {table} WHERE NOT truncated GROUP BY width ORDER BY width;
""")
    checks['lignes_avec_composant_brut_vide'] = sum(row['lignes_avec_composant_brut_vide'] for row in widths)
    daily = typed_rows(exe, f"""
SELECT cast(t AS DATE) AS jour, count(*) AS ticks, min(t) AS premier_tick, max(t) AS dernier_tick
FROM {table} WHERE NOT truncated AND t IS NOT NULL GROUP BY 1 ORDER BY 1;
""")
    covered_days = {date.fromisoformat(row['jour']) for row in daily}
    first, last = min(covered_days), max(covered_days)
    days = (first + timedelta(days=offset) for offset in range((last - first).days + 1))
    missing = [day.isoformat() for day in days if day.weekday() < 5 and day not in covered_days]
    gaps = typed_rows(exe, f"""
WITH gaps AS (
    SELECT t, previous_t, date_diff('millisecond', previous_t, t) AS gap_ms
    FROM {table} WHERE NOT truncated AND t > previous_t
)
SELECT count(*) FILTER (WHERE gap_ms >= 3600000 AND cast(t AS DATE) = cast(previous_t AS DATE)) AS intra_jour_ge_1h,
       count(*) FILTER (WHERE gap_ms >= 86400000) AS ge_24h,
       count(*) FILTER (WHERE gap_ms >= 259200000) AS ge_72h,
       max(gap_ms) AS max_ms
FROM gaps;
""")[0]
    gaps['principaux'] = typed_rows(exe, f"""
SELECT row_id + 1 AS ligne_csv, previous_t AS depuis, t AS jusque,
       date_diff('millisecond', previous_t, t) AS gap_ms,
       cast(t AS DATE) = cast(previous_t AS DATE) AS intra_jour
FROM {table} WHERE NOT truncated AND t > previous_t ORDER BY gap_ms DESC, ligne_csv LIMIT 25;
""")
    gaps['exemples_intra_jour'] = typed_rows(exe, f"""
SELECT row_id + 1 AS ligne_csv, previous_t AS depuis, t AS jusque,
       date_diff('millisecond', previous_t, t) AS gap_ms
FROM {table} WHERE NOT truncated AND cast(t AS DATE) = cast(previous_t AS DATE)
AND date_diff('millisecond', previous_t, t) >= 3600000
ORDER BY gap_ms DESC, ligne_csv LIMIT 25;
""")
    reversals = typed_rows(exe, f"""
SELECT row_id + 1 AS ligne_csv, previous_t AS precedent, t AS suivant,
       date_diff('millisecond', previous_t, t) AS delta_ms
FROM {table} WHERE NOT truncated AND t < previous_t ORDER BY ligne_csv LIMIT 25;
""")
    print(f'{source.name}: coverage and field checks completed', flush=True)
    volumes = {name: volume_stats(exe, table, name) for name in ('ask_volume', 'bid_volume')}
    print(f'{source.name}: volume distributions completed', flush=True)
    strict = (checks['reculs_adjacent'] == checks['egalites_adjacentes'] == checks['horodatages_invalides'] == 0
              and checks['paires_adjacentes_lisibles'] == physical_complete - 1
              and checks['lignes_ignorees_par_lecteur'] == 0)
    if strict:
        zero = dict.fromkeys(('copies_identiques_excedentaires', 'groupes_lignes_identiques',
                              'horodatages_repetes', 'horodatages_avec_lignes_distinctes'), 0)
        repeated = {'totaux': zero, 'par_mois': [{'mois': p.name[5:]} | zero for p in sorted(partitions.glob('mois=*'))],
                    'methode': 'unicite_prouvee_par_horodatages_strictement_croissants'}
    else:
        repeated = duplicates(exe, partitions)
        repeated['methode'] = 'groupement_exact_par_horodatage_et_ligne_brute'
    print(f'{source.name}: exact duplicates completed', flush=True)
    return {'fichier': source.name, 'octets': source.stat().st_size, 'controle': checks,
            'format': widths, 'couverture_annuelle': per_calendar(daily, 4),
            'couverture_mensuelle': per_calendar(daily, 7), 'couverture_journaliere': daily,
            'lun_ven_sans_ticks': missing, 'intervalles': gaps, 'exemples_reculs': reversals,
            'volumes': volumes, 'doublons': repeated}


def profiler(chemin, dossier_temporaire):
    nombre, fragment = physical_layout(chemin)
    with chdir(dossier_temporaire):
        partitions = dossier_temporaire / 'partitions'
        if chemin.stat().st_size >= 10 * 1024**3:
            stage_large(duckdb, chemin, partitions, fragment)
        else:
            stage(duckdb, chemin, partitions, fragment)
        return summarize(duckdb, partitions, chemin, nombre, fragment)



def charger_resultats_integres():
    payload = ''.join(meta_audit['audit_base64_zlib'])
    return json.loads(zlib.decompress(base64.b64decode(payload)))


print('Contrôles chargés : couverture, valeurs manquantes, prix, spreads, volumes, doublons et ordre.')


Contrôles chargés : couverture, valeurs manquantes, prix, spreads, volumes, doublons et ordre.


In [7]:
profils = {} if mode == 'complet' else charger_resultats_integres()
if mode != 'snapshot':
    for nom in FILES.values():
        if not (data_dir / nom).is_file():
            raise FileNotFoundError(data_dir / nom)
    if not duckdb.is_file():
        raise FileNotFoundError(duckdb)
    scratch.mkdir(parents=True, exist_ok=True)
if mode == 'complet':
    version_duckdb = sql_rows(duckdb, 'SELECT version() AS version;')[0]['version']
    for nom, fichier in FILES.items():
        print(f'Calcul de {nom} avec DuckDB {version_duckdb}…', flush=True)
        with TemporaryDirectory(prefix=f'xauusd_{nom}_', dir=scratch) as temp:
            profils[nom] = profiler(data_dir / fichier, Path(temp)) | {'duckdb': version_duckdb}
tableau(['Export', 'DuckDB', 'Lignes complètes disponibles'], [
    (nom, profils[nom]['duckdb'], profils[nom]['controle']['lignes_terminees']) for nom in FILES
])
afficher('Le profil chargé alimente tous les tableaux de contrôle. Le mode complet le recalcule avec le binaire DuckDB local.')


| Export | DuckDB | Lignes complètes disponibles |
| --- | --- | --- |
| ancien | v2.0.1 | 432 697 435 |
| recent | v2.0.1 | 116 077 131 |

Le profil chargé alimente tous les tableaux de contrôle. Le mode complet le recalcule avec le binaire DuckDB local.

## Comparaison des deux CSV historiques

Le test compare chaque ligne brute à la ligne de même position sur la période commune. Le fragment terminal de l’ancien fichier est vérifié séparément.

In [8]:
TAILLE_DATE = 23
TAILLE_BLOC = 8 * 1024 * 1024


def trouver_debut_recouvrement(ancien, debut_ancien, premiere_date_recente):
    """Cherche le premier tick ancien daté du début du second fichier ou après."""
    gauche = debut_ancien
    droite = ancien.seek(0, 2)

    # On cherche d'abord par position dans le fichier pour éviter de relire 25 Go.
    while droite - gauche > TAILLE_BLOC:
        milieu = (gauche + droite) // 2
        ancien.seek(milieu - 1)
        ancien.readline()  # Repart au début d'une ligne entière.
        ligne = ancien.readline()
        if ligne and ligne[:TAILLE_DATE] < premiere_date_recente:
            gauche = ancien.tell()
        else:
            droite = milieu

    ancien.seek(gauche)
    position = gauche
    while ligne := ancien.readline():
        if ligne[:TAILLE_DATE] >= premiere_date_recente:
            return position
        position = ancien.tell()
    return position


def scan_brut(chemin, *, entete):
    return (f"read_csv({sql_string(chemin.as_posix())}, columns={{'raw':'VARCHAR'}}, "
            f"delim='|', quote='', escape='', header={str(entete).lower()}, auto_detect=false, parallel=false)")


def comparer(ancien, recent, temp):
    extrait = temp / 'fin_ancien.txt'
    with ancien.open('rb') as a, recent.open('rb') as b, extrait.open('wb') as sortie:
        def entete(f):
            if f.readline().removeprefix(b'\xef\xbb\xbf').rstrip(b'\r\n') != HEADER:
                raise ValueError(f'En-tête inattendu : {f.name}')
            return f.tell()

        debut_ancien = entete(a)
        debut_recent = entete(b)
        debut = trouver_debut_recouvrement(a, debut_ancien, b.readline()[:TAILLE_DATE])
        fin = a.seek(0, 2)
        a.seek(max(0, fin - 4096))
        fragment = a.read().rsplit(b'\n', 1)[-1]
        fin -= len(fragment)
        a.seek(debut)
        lignes = 0
        while a.tell() < fin:
            bloc = a.read(min(TAILLE_BLOC, fin - a.tell()))
            if not bloc:
                raise EOFError(ancien)
            sortie.write(bloc)
            lignes += bloc.count(b'\n')
        b.seek(debut_recent + fin - debut)
        suivant = b.readline().rstrip(b'\r\n')
    if lignes == 0:
        commun = {'identiques': 0, 'differentes': 0, 'premier': None, 'dernier': None}
    else:
        commun = typed_rows(duckdb, f"""
SELECT count(*) FILTER (WHERE a.raw = b.raw) AS identiques,
       count(*) FILTER (WHERE a.raw IS DISTINCT FROM b.raw) AS differentes,
       min(left(a.raw, 23)) AS premier, max(left(a.raw, 23)) AS dernier
FROM {scan_brut(extrait, entete=False)} a
POSITIONAL JOIN (SELECT raw FROM {scan_brut(recent, entete=True)} LIMIT {lignes}) b;
""")[0]
    recupere = bool(fragment and suivant.startswith(fragment) and commun['differentes'] == 0)
    return commun | {'fragment_confirme': recupere, 'fragment_octets': len(fragment),
                     'tick_recupere': suivant[:23].decode('ascii') if recupere else None}


if mode != 'snapshot':
    with TemporaryDirectory(prefix='xauusd_comparaison_', dir=scratch) as chemin, chdir(chemin):
        profils['comparaison'] = comparer(*(data_dir / nom for nom in FILES.values()), Path(chemin))

comparaison = profils['comparaison']
tableau(['Chevauchement', 'Résultat'], [(k, 'oui' if v is True else 'non' if v is False else v)
                                     for k, v in comparaison.items()])
afficher('Les lignes identiques de la période commune expliquent le double comptage des deux CSV. Le fragment terminal est contrôlé séparément.')


| Chevauchement | Résultat |
| --- | --- |
| identiques | 43 478 752 |
| differentes | 0 |
| premier | 2021.01.04 01:00:00.413 |
| dernier | 2021.10.15 23:54:59.188 |
| fragment_confirme | oui |
| fragment_octets | 53 |
| tick_recupere | 2021.10.15 23:55:00.082 |

Les lignes identiques de la période commune expliquent le double comptage des deux CSV. Le fragment terminal est contrôlé séparément.

## Nombre de ticks

In [9]:
mesures = [('Lignes complètes', 'lignes_terminees'), ('Lignes physiques hors en-tête', 'lignes_physiques_donnees'), ('Fragment terminal', 'finales_tronquees'), ('Premier tick', 'premier_tick'), ('Dernier tick complet', 'dernier_tick')]
tableau(['Mesure', 'Ancien', 'Récent'], [(label, profils['ancien']['controle'].get(k), profils['recent']['controle'].get(k)) for label, k in mesures])
afficher('Les lignes complètes donnent le nombre de ticks comptabilisés. Un fragment en fin de fichier est signalé séparément.')


| Mesure | Ancien | Récent |
| --- | --- | --- |
| Lignes complètes | 432 697 435 | 116 077 131 |
| Lignes physiques hors en-tête | 432 697 436 | 116 077 131 |
| Fragment terminal | 1 | 0 |
| Premier tick | 2004-01-01 02:00:02.937 | 2021-01-04 01:00:00.413 |
| Dernier tick complet | 2021-10-15 23:54:59.188 | 2026-11-20 06:00:02.344 |

Les lignes complètes donnent le nombre de ticks comptabilisés. Un fragment en fin de fichier est signalé séparément.

## Valeurs nulles

In [10]:
mesures = [('Horodatages invalides ou absents', 'horodatages_invalides'), ('Lignes avec un composant explicitement vide', 'lignes_avec_composant_brut_vide'), ('Ask vide (9 composants)', 'ask_vide'), ('Bid vide (9 composants)', 'bid_vide'), ('Volume Ask vide (9 composants)', 'ask_volume_vide'), ('Volume Bid vide (9 composants)', 'bid_volume_vide')]
tableau(['Mesure', 'Ancien', 'Récent'], [(label, profils['ancien']['controle'].get(k), profils['recent']['controle'].get(k)) for label, k in mesures])
afficher('Les valeurs absentes et les champs dont le découpage est ambigu sont comptés séparément.')


| Mesure | Ancien | Récent |
| --- | --- | --- |
| Horodatages invalides ou absents | 0 | 0 |
| Lignes avec un composant explicitement vide | 0 | 0 |
| Ask vide (9 composants) | 0 | 0 |
| Bid vide (9 composants) | 0 | 0 |
| Volume Ask vide (9 composants) | 0 | 0 |
| Volume Bid vide (9 composants) | 0 | 0 |

Les valeurs absentes et les champs dont le découpage est ambigu sont comptés séparément.

## Prix invalides

In [11]:
mesures = [('Ask lisibles', 'ask_lisible'), ('Bid lisibles', 'bid_lisible'), ('Ask ≤ 0', 'ask_non_positifs'), ('Bid ≤ 0', 'bid_non_positifs'), ('Ask minimal', 'ask_minimum'), ('Bid minimal', 'bid_minimum')]
tableau(['Mesure', 'Ancien', 'Récent'], [(label, profils['ancien']['controle'].get(k), profils['recent']['controle'].get(k)) for label, k in mesures])
afficher('Les tests de prix ne portent que sur les champs lisibles. Un prix inférieur ou égal à zéro est incompatible avec une cotation normale.')


| Mesure | Ancien | Récent |
| --- | --- | --- |
| Ask lisibles | 431 959 749 | 115 973 989 |
| Bid lisibles | 431 959 749 | 115 973 989 |
| Ask ≤ 0 | 0 | 0 |
| Bid ≤ 0 | 0 | 0 |
| Ask minimal | 371,032 | 1615,092 |
| Bid minimal | 370,627 | 1614,71 |

Les tests de prix ne portent que sur les champs lisibles. Un prix inférieur ou égal à zéro est incompatible avec une cotation normale.

## Spread négatif

In [12]:
mesures = [('Paires lisibles', 'paires_de_prix_lisibles'), ('Ask < Bid', 'spread_negatif'), ('Ask = Bid', 'spread_nul'), ('Spread minimal', 'spread_minimum')]
tableau(['Mesure', 'Ancien', 'Récent'], [(label, profils['ancien']['controle'].get(k), profils['recent']['controle'].get(k)) for label, k in mesures])
afficher('Le spread est Ask moins Bid. Un spread négatif signale une cotation à examiner.')


| Mesure | Ancien | Récent |
| --- | --- | --- |
| Paires lisibles | 431 959 749 | 115 973 989 |
| Ask < Bid | 0 | 0 |
| Ask = Bid | 0 | 0 |
| Spread minimal | 0,001 | 0,001 |

Le spread est Ask moins Bid. Un spread négatif signale une cotation à examiner.

## Ordre chronologique

In [13]:
mesures = [('Reculs adjacents', 'reculs_adjacent'), ('Reculs avec changement de date', 'reculs_changement_date'), ('Horodatages adjacents égaux', 'egalites_adjacentes'), ('Plus petit intervalle adjacent (ms)', 'pire_recul_ms')]
tableau(['Mesure', 'Ancien', 'Récent'], [(label, profils['ancien']['controle'].get(k), profils['recent']['controle'].get(k)) for label, k in mesures])
afficher('Un recul survient quand le tick suivant a un horodatage antérieur. Une égalité peut correspondre à plusieurs cotations à la même milliseconde.')


| Mesure | Ancien | Récent |
| --- | --- | --- |
| Reculs adjacents | 0 | 0 |
| Reculs avec changement de date | 0 | 0 |
| Horodatages adjacents égaux | 0 | 0 |
| Plus petit intervalle adjacent (ms) | 1 | 1 |

Un recul survient quand le tick suivant a un horodatage antérieur. Une égalité peut correspondre à plusieurs cotations à la même milliseconde.

## Couverture annuelle

In [14]:
tableau(['Export', 'Année', 'Ticks', 'Jours avec ticks', 'Jours dans la fenêtre', 'Lun–ven sans ticks'], [
    (nom, row['periode'], row['ticks'], row['jours_couverts'], row['jours_dans_fenetre'], row['lun_ven_sans_ticks'])
    for nom in FILES for row in profils[nom]['couverture_annuelle']
])
afficher('Les effectifs annuels permettent de repérer les années les moins couvertes et les bornes partielles.')


| Export | Année | Ticks | Jours avec ticks | Jours dans la fenêtre | Lun–ven sans ticks |
| --- | --- | --- | --- | --- | --- |
| ancien | 2004 | 7 306 572 | 263 | 366 | 0 |
| ancien | 2005 | 10 443 868 | 261 | 365 | 0 |
| ancien | 2006 | 12 791 473 | 261 | 365 | 0 |
| ancien | 2007 | 5 350 902 | 261 | 365 | 0 |
| ancien | 2008 | 3 744 573 | 262 | 366 | 0 |
| ancien | 2009 | 4 096 575 | 261 | 365 | 0 |
| ancien | 2010 | 13 100 183 | 261 | 365 | 0 |
| ancien | 2011 | 17 556 744 | 259 | 365 | 1 |
| ancien | 2012 | 23 439 525 | 260 | 366 | 1 |
| ancien | 2013 | 23 461 967 | 259 | 365 | 2 |
| ancien | 2014 | 21 561 539 | 258 | 365 | 3 |
| ancien | 2015 | 25 767 375 | 258 | 365 | 3 |
| ancien | 2016 | 46 200 033 | 258 | 366 | 3 |
| ancien | 2017 | 45 836 980 | 257 | 365 | 3 |
| ancien | 2018 | 35 920 345 | 258 | 365 | 3 |
| ancien | 2019 | 36 812 859 | 258 | 365 | 3 |
| ancien | 2020 | 55 827 170 | 259 | 366 | 3 |
| ancien | 2021 | 43 478 752 | 204 | 288 | 2 |
| recent | 2021 | 53 099 638 | 258 | 362 | 2 |
| recent | 2022 | 53 838 933 | 258 | 365 | 2 |
| recent | 2023 | 7 205 128 | 83 | 365 | 177 |
| recent | 2024 | 1 596 736 | 85 | 366 | 177 |
| recent | 2025 | 314 006 | 40 | 365 | 221 |
| recent | 2026 | 22 690 | 34 | 324 | 198 |

Les effectifs annuels permettent de repérer les années les moins couvertes et les bornes partielles.

## Couverture mensuelle

In [15]:
tableau(['Export', 'Mois dans la fenêtre', 'Mois absents', 'Liste'], [
    (nom, len(profils[nom]['couverture_mensuelle']), len(absents), ', '.join(absents) or 'Aucun')
    for nom in FILES for absents in [[row['periode'] for row in profils[nom]['couverture_mensuelle'] if not row['ticks']]]
])

# Changer l'année pour consulter les effectifs mensuels de chaque export.
annee = '2021'
mois = [(nom, row['periode'], row['ticks'], row['jours_couverts']) for nom in FILES
        for row in profils[nom]['couverture_mensuelle'] if row['periode'].startswith(annee)]
tableau(['Export', 'Mois', 'Ticks', 'Jours couverts'], mois or [('—', annee, 'Hors couverture', '—')])
afficher('Ce tableau indique les mois sans observation dans la fenêtre couverte par chaque export.')


| Export | Mois dans la fenêtre | Mois absents | Liste |
| --- | --- | --- | --- |
| ancien | 214 | 0 | Aucun |
| recent | 71 | 17 | 2023-06, 2023-07, 2023-08, 2023-12, 2024-01, 2024-05, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2026-07, 2026-08, 2026-09, 2026-10 |

| Export | Mois | Ticks | Jours couverts |
| --- | --- | --- | --- |
| ancien | 2021-01 | 4 735 644 | 20 |
| ancien | 2021-02 | 5 269 851 | 20 |
| ancien | 2021-03 | 6 485 487 | 23 |
| ancien | 2021-04 | 4 333 612 | 21 |
| ancien | 2021-05 | 4 705 096 | 21 |
| ancien | 2021-06 | 4 511 805 | 22 |
| ancien | 2021-07 | 3 465 635 | 22 |
| ancien | 2021-08 | 3 455 886 | 22 |
| ancien | 2021-09 | 4 367 511 | 22 |
| ancien | 2021-10 | 2 148 225 | 11 |
| recent | 2021-01 | 4 735 644 | 20 |
| recent | 2021-02 | 5 269 851 | 20 |
| recent | 2021-03 | 6 485 487 | 23 |
| recent | 2021-04 | 4 333 612 | 21 |
| recent | 2021-05 | 4 705 096 | 21 |
| recent | 2021-06 | 4 511 805 | 22 |
| recent | 2021-07 | 3 465 635 | 22 |
| recent | 2021-08 | 3 455 886 | 22 |
| recent | 2021-09 | 4 367 511 | 22 |
| recent | 2021-10 | 4 171 280 | 21 |
| recent | 2021-11 | 4 156 708 | 22 |
| recent | 2021-12 | 3 441 123 | 22 |

Ce tableau indique les mois sans observation dans la fenêtre couverte par chaque export.

## Couverture journalière et interruptions

Un jour ouvré sans tick ou un long intervalle signale une période à examiner. Les mesures seules permettent de constater la coupure. Un calendrier de la source serait nécessaire pour lui attribuer une cause.

In [16]:
tableau(['Export', 'Jours avec ticks', 'Lun–ven sans ticks', 'Intra-jour ≥ 1 h', 'Intervalles ≥ 24 h', 'Intervalles ≥ 72 h'], [
    (nom, len(profils[nom]['couverture_journaliere']), len(profils[nom]['lun_ven_sans_ticks']),
     *(profils[nom]['intervalles'][k] for k in ('intra_jour_ge_1h', 'ge_24h', 'ge_72h')))
    for nom in FILES
])
tableau(['Export', 'Avant', 'Après', 'Durée (jours)'], [
    (nom, row['depuis'], row['jusque'], round(row['gap_ms'] / 86400000, 3))
    for nom in FILES for row in profils[nom]['intervalles']['principaux'][:5]
])
# Changer cette date pour consulter un jour précis dans les deux exports.
jour = '2021-01-04'
tableau(['Export', 'Jour', 'Ticks'], [(nom, jour, next((r['ticks'] for r in profils[nom]['couverture_journaliere']
                                                     if r['jour'] == jour), 0)) for nom in FILES])
afficher('Les coupures signalent des périodes à examiner. Le calendrier de marché est nécessaire pour distinguer fermeture et lacune.')


| Export | Jours avec ticks | Lun–ven sans ticks | Intra-jour ≥ 1 h | Intervalles ≥ 24 h | Intervalles ≥ 72 h |
| --- | --- | --- | --- | --- | --- |
| ancien | 4 618 | 27 | 35 | 937 | 18 |
| recent | 758 | 777 | 20 | 174 | 37 |

| Export | Avant | Après | Durée (jours) |
| --- | --- | --- | --- |
| ancien | 2015-12-24 20:30:00.949 | 2015-12-28 01:01:11.385 | 3,188 |
| ancien | 2020-12-24 20:44:31.208 | 2020-12-28 01:00:00.315 | 3,177 |
| ancien | 2016-03-24 22:59:26.661 | 2016-03-28 01:00:00.559 | 3,084 |
| ancien | 2013-03-28 22:59:50.776 | 2013-04-01 01:00:04.921 | 3,083 |
| ancien | 2019-04-18 23:58:00.773 | 2019-04-22 01:00:06.343 | 3,043 |
| recent | 2025-04-29 05:00:08.58 | 2025-12-12 02:00:00.063 | 226,875 |
| recent | 2026-06-05 05:00:02.204 | 2026-11-02 02:00:00.009 | 149,875 |
| recent | 2023-05-24 08:00:22.038 | 2023-09-19 14:00:04.149 | 118,25 |
| recent | 2023-11-01 03:00:04.138 | 2024-02-26 07:00:00.081 | 117,167 |
| recent | 2024-04-30 18:00:50.899 | 2024-06-24 01:00:01.576 | 54,291 |

| Export | Jour | Ticks |
| --- | --- | --- |
| ancien | 2021-01-04 | 218 880 |
| recent | 2021-01-04 | 218 880 |

Les coupures signalent des périodes à examiner. Le calendrier de marché est nécessaire pour distinguer fermeture et lacune.

## Volumes

Les percentiles sont calculés par fréquence cumulée sur les champs lisibles. Les effectifs des classes et l’écart-type sont consultables dans `profils[export]['volumes']`.

In [17]:
colonnes = ['observations', 'negatifs', 'zeros', 'minimum', 'p01', 'p25', 'p50', 'p75', 'p95', 'p99', 'maximum', 'moyenne']
tableau(['Export', 'Volume', *colonnes], [(nom, direction, *(stats[k] for k in colonnes))
        for nom in FILES for direction, stats in profils[nom]['volumes'].items()])
afficher('Les percentiles décrivent la distribution des volumes lisibles pour chaque côté de la cotation.')


| Export | Volume | observations | negatifs | zeros | minimum | p01 | p25 | p50 | p75 | p95 | p99 | maximum | moyenne |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ancien | ask_volume | 431 959 749 | 0 | 0 | 1e-05 | 0,0001 | 0,00022 | 0,00037 | 0,00044 | 0,0008 | 0,0012 | 0,82 | 0,00040344501 |
| ancien | bid_volume | 431 959 749 | 0 | 0 | 1e-05 | 0,0001 | 0,00022 | 0,00037 | 0,00043 | 0,0008 | 0,0012 | 0,37 | 0,00040333057 |
| recent | ask_volume | 115 973 989 | 0 | 0 | 1e-05 | 0,0001 | 0,00015 | 0,00018 | 0,00037 | 0,00045 | 0,00067 | 0,0157 | 0,00024679148 |
| recent | bid_volume | 115 973 989 | 0 | 0 | 1e-05 | 0,0001 | 0,00015 | 0,00019 | 0,00037 | 0,00045 | 0,00067 | 0,0135 | 0,0002487046 |

Les percentiles décrivent la distribution des volumes lisibles pour chaque côté de la cotation.

## Doublons internes

Une répétition complète exige le même horodatage. Si les horodatages sont strictement croissants, les doublons sont impossibles. Sinon, le moteur regroupe les lignes brutes et distingue les horodatages dont les cotations diffèrent.

In [18]:
labels = {'copies_identiques_excedentaires': 'Copies identiques au-delà de la première',
          'groupes_lignes_identiques': 'Groupes de lignes identiques',
          'horodatages_repetes': 'Horodatages répétés',
          'horodatages_avec_lignes_distinctes': 'Horodatages avec plusieurs lignes distinctes'}
tableau(['Mesure', 'Ancien', 'Récent'], [(label, profils['ancien']['doublons']['totaux'][k],
         profils['recent']['doublons']['totaux'][k]) for k, label in labels.items()])
afficher('Les doublons exacts et les cotations différentes partageant le même horodatage sont comptés séparément.')


| Mesure | Ancien | Récent |
| --- | --- | --- |
| Copies identiques au-delà de la première | 0 | 0 |
| Groupes de lignes identiques | 0 | 0 |
| Horodatages répétés | 0 | 0 |
| Horodatages avec plusieurs lignes distinctes | 0 | 0 |

Les doublons exacts et les cotations différentes partageant le même horodatage sont comptés séparément.

## Format décimal

Le format standard présente l’horodatage et quatre nombres à virgule, soit neuf composants après découpage. Un nombre entier peut changer la largeur et rendre le découpage ambigu. Ces lignes restent dans les contrôles temporels et les comptes bruts.

In [19]:
tableau(['Export', 'Composants', 'Lignes', 'Exemple brut'], [(nom, r['composants'], r['lignes'], r['exemple'])
        for nom in FILES for r in profils[nom]['format']])
tableau(['Export', 'Structure standard', 'Autre structure'], [(nom, profils[nom]['controle']['format_standard'],
        profils[nom]['controle']['format_non_standard']) for nom in FILES])
afficher('Les largeurs atypiques révèlent les lignes dont les virgules ne permettent pas un découpage certain en quatre valeurs.')


| Export | Composants | Lignes | Exemple brut |
| --- | --- | --- | --- |
| ancien | 6 | 5 | 2015.07.20 12:31:00.099,1115,1114,669,0,0 |
| ancien | 7 | 1 364 | 2010.12.27 01:30:39.937,1373,1372,0,00029,0,00021 |
| ancien | 8 | 736 317 | 2004.01.01 18:30:41.511,415,414,494,0,00095,0,00022 |
| ancien | 9 | 431 959 749 | 2004.01.01 02:00:02.937,415,332,414,922,0,00048,0,00026 |
| recent | 8 | 103 142 | 2021.01.04 01:01:22.487,1909,25,1908,0,00037,0,00037 |
| recent | 9 | 115 973 989 | 2021.01.04 01:00:00.413,1905,366,1904,998,0,00037,0,00015 |

| Export | Structure standard | Autre structure |
| --- | --- | --- |
| ancien | 431 959 749 | 737 686 |
| recent | 115 973 989 | 103 142 |

Les largeurs atypiques révèlent les lignes dont les virgules ne permettent pas un découpage certain en quatre valeurs.

## Bilan du profilage des CSV historiques

Les lignes communes identiques sont retirées du calcul de référence. Le fichier fusionné appartient à la phase suivante.

In [20]:
total = sum(profils[nom]['controle']['lignes_terminees'] for nom in FILES)
communs = profils['comparaison']['identiques']
ecarts = profils['comparaison']['differentes']
total_historique = total - communs if ecarts == 0 else 'À vérifier'
tableau(['Indicateur', 'Valeur'], [('Lignes complètes dans les deux sources', total),
        ('Lignes communes identiques', communs), ('Écarts sur le chevauchement', ecarts),
        ('Lignes historiques après retrait du chevauchement', total_historique)])
afficher("Cet effectif couvre les deux CSV historiques. Les archives complémentaires sont examinées dans le bilan de récupération.")


| Indicateur | Valeur |
| --- | --- |
| Lignes complètes dans les deux sources | 548 774 566 |
| Lignes communes identiques | 43 478 752 |
| Écarts sur le chevauchement | 0 |
| Lignes historiques après retrait du chevauchement | 505 295 814 |

Cet effectif couvre les deux CSV historiques. Les archives complémentaires sont examinées dans le bilan de récupération.

## Data Cleaning & Recovery · prochain chantier

Les tableaux restituent les preuves disponibles et les règles à appliquer lors de cette phase.

In [21]:
tableau(['Récupération', 'Minutes absentes sous UTC+2 et UTC+3', 'Ticks dans ces minutes'], [
    (lot['year'], lot['absent_minutes'], lot['ticks_in_absent_minutes']) for lot in preuves['recent']
])
afficher("Ces minutes apparaissent dans les archives UTC et sont absentes sous les deux décalages locaux testés. Elles forment un périmètre conservateur de récupération.")

| Récupération | Minutes absentes sous UTC+2 et UTC+3 | Ticks dans ces minutes |
| --- | --- | --- |
| 2 023 | 289 735 | 29 168 532 |
| 2 024 | 345 334 | 54 425 199 |
| 2 025 | 352 684 | 71 025 659 |
| 2 026 | 316 158 | 84 274 006 |

Ces minutes apparaissent dans les archives UTC et sont absentes sous les deux décalages locaux testés. Elles forment un périmètre conservateur de récupération.

In [22]:
decimales = preuves['decimals']
tableau(['Reconstruction décimale · pilote du 5 janvier 2021', 'Lignes'], [
    ('Lignes ambiguës appariées à JForex', decimales['ambiguous_apparied']),
    ('Reconstitutions exactes avec une seule solution', decimales['ambiguous_exact_unique']),
    ('Ligne appariée non résolue', decimales['ambiguous_unresolved']),
    ('Lignes ambiguës sans référence', decimales['ambiguous_without_reference']),
    ('Divergences de volume', decimales['volume_differences']),
])
afficher("Une reconstruction documentée conserve le tick d’origine dans le décompte. Les volumes divergents restent à rapprocher de la source.")

| Reconstruction décimale · pilote du 5 janvier 2021 | Lignes |
| --- | --- |
| Lignes ambiguës appariées à JForex | 122 |
| Reconstitutions exactes avec une seule solution | 121 |
| Ligne appariée non résolue | 1 |
| Lignes ambiguës sans référence | 3 |
| Divergences de volume | 36 |

Une reconstruction documentée conserve le tick d’origine dans le décompte. Les volumes divergents restent à rapprocher de la source.

In [23]:
tableau(['Conversion UTC · journée testée', 'Horloge locale moins UTC', 'Horodatages appariés'], [
    (jour, f'UTC+{decalage}', ticks) for jour, decalage, ticks in preuves['utc_examples']
])
afficher("Les sondes confirment les deux décalages selon la date. La conversion de tout l’historique exige une règle de fuseau vérifiée par période.")

| Conversion UTC · journée testée | Horloge locale moins UTC | Horodatages appariés |
| --- | --- | --- |
| 2023-02-02 | UTC+2 | 216 869 |
| 2024-09-19 | UTC+3 | 292 710 |
| 2025-12-12 | UTC+2 | 222 718 |
| 2026-01-02 | UTC+2 | 90 |

Les sondes confirment les deux décalages selon la date. La conversion de tout l’historique exige une règle de fuseau vérifiée par période.

In [24]:
recuperes = sum(lot['ticks_in_absent_minutes'] for lot in preuves['recent'])
base_auditee = preuves['baseline']
jforex_partiel = preuves['jforex']['ticks']
tableau(['Fusion sans duplication · bilan conservateur', 'Lignes'], [
    ('Base historique après retrait du chevauchement', base_auditee),
    ('Ticks dans des minutes absentes des CSV historiques', recuperes),
    ('JForex partiel du 23 novembre 2026', jforex_partiel),
    ('Total conservateur avant fusion physique', base_auditee + recuperes + jforex_partiel),
])
afficher("La fusion finale devra comparer les cotations avec leur provenance et garder les ticks distincts qui partagent la même milliseconde. Les comptes HistData restent propres à leur couverture.")

| Fusion sans duplication · bilan conservateur | Lignes |
| --- | --- |
| Base historique après retrait du chevauchement | 505 295 814 |
| Ticks dans des minutes absentes des CSV historiques | 238 893 396 |
| JForex partiel du 23 novembre 2026 | 500 254 |
| Total conservateur avant fusion physique | 744 689 464 |

La fusion finale devra comparer les cotations avec leur provenance et garder les ticks distincts qui partagent la même milliseconde. Les comptes HistData restent propres à leur couverture.

In [25]:
tableau(['Quarantaine · preuve conservée', 'Présence locale'], [
    (chemin, 'Présente' if (data_dir / chemin).exists() else 'Hors du dossier local')
    for chemin in preuves['quarantine']
])
afficher("La quarantaine garde les exports dont l’horloge ou la provenance reste à confirmer. Le prochain chantier consiste à valider l’UTC, les reconstructions et les conflits avant d’écrire la fusion.")

| Quarantaine · preuve conservée | Présence locale |
| --- | --- |
| quarantine/XAUUSD_2025-08-05_HistData_UTC_plus5_non_valide.csv | Présente |
| quarantine/esmaeil_ticks_pilote | Présente |

La quarantaine garde les exports dont l’horloge ou la provenance reste à confirmer. Le prochain chantier consiste à valider l’UTC, les reconstructions et les conflits avant d’écrire la fusion.